# Ingest a corpus on GPU, publish the index

Embedding a corpus is a GPU-bound one-time batch job; serving queries is a CPU-bound latency job. This notebook does the first half and publishes the result, so a free CPU Space can pull a prebuilt index instead of spending an hour re-embedding.

**Runtime → Change runtime type → GPU** before running.

The same code runs on CPU — `device=auto` resolves whatever is present — it is just slower.

In [ ]:
!git clone https://github.com/YOUR_USER/YOUR_REPO.git vsearch_repo
%cd vsearch_repo
# Colab already ships a CUDA torch; installing the project's pinned CPU build
# would replace it. --no-deps for torch keeps Colab's GPU wheel in place.
!pip install -q -e . 2>&1 | tail -2

In [ ]:
import torch

print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

## Credentials

`HF_TOKEN` is needed to publish the index, and to load DINOv3 (licence-gated — accept it at https://huggingface.co/facebook/dinov3-vits16-pretrain-lvd1689m). Without it the encoder registry falls back to the ungated `dinov2-small` and logs the substitution.

Store it in Colab **Secrets** (key icon, left sidebar) rather than pasting it into a cell — notebook outputs get committed.

In [ ]:
import os

from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["VSEARCH_DEVICE"] = "auto"
os.environ["VSEARCH_BATCH_SIZE"] = "128"  # raise on GPU; 8-16 is right on a laptop CPU

ARTIFACT_REPO = "YOUR_USER/vsearch-index"  # a *dataset* repo: this is data, not weights

## Ingest

Safe to re-run: work commits in shards and a restart resumes from the first missing one. Colab disconnects mid-run are expected, not fatal.

In [ ]:
# Demo corpus: 44k product images with eight filterable facets.
!vsearch ingest --corpus fashion --encoder clip --shard-size 2048

In [ ]:
# Image->image quality comes from the self-supervised encoder.
# Falls back to dinov2-small if the DINOv3 licence is not accepted.
!vsearch ingest --corpus fashion --encoder dinov3 --shard-size 2048

In [ ]:
# Evaluation corpus. `flickr1k` is the canonical 1000-image test split
# published standalone (142 MB). Reaching the same images through the full
# 4.31 GB flickr30k corpus means streaming all of it to keep 3%, so only use
# `--corpus flickr30k --split test` if you specifically want the larger index.
!vsearch ingest --corpus flickr1k --encoder clip --shard-size 250

## Evaluate

Text→image against real human captions: 1000 images, 5 captions each = 5000 queries.

In [ ]:
# Text->image against real human captions: 1000 images, 5 captions each.
# --control re-runs with captions shuffled onto the wrong images; it must land
# at chance (1/1000), which is what rules out a leaking harness.
!python -m vsearch.eval.run_eval --corpus flickr1k --encoder clip --protocol text --split test --k 10 --control

In [ ]:
!python -m vsearch.eval.run_eval --corpus fashion --encoder clip --encoder dinov3 --protocol image --k 10

## Benchmark on GPU

Fills in the GPU rows the laptop run could not produce.

In [ ]:
!pip install -q onnxscript onnxruntime psutil
!python -m vsearch.bench.run_bench --encoder clip --batch 1 --batch 32 --runs 20

## Publish

Uploads the index, metadata and thumbnails — but not the intermediate shards, since `IndexFlatIP` already stores the vectors and shipping both would roughly double the artifact.

In [ ]:
!vsearch publish --corpus fashion --encoder clip --repo $ARTIFACT_REPO
!vsearch publish --corpus fashion --encoder dinov3 --repo $ARTIFACT_REPO

## Then

Set `VSEARCH_ARTIFACT_REPO` on the Space to this repo id. Its entrypoint pulls the index at startup and serves it — see `deploy/README.md`.